In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd().parents[1]
sys.path.append(str(project_root))

from src.data_prep import to_snake_case, unzip_csv

# Diabetes prediction - Cleanup (BRFSS)

This notebook reproduces the original BRFSS2015 diabetes dataset preparation methodology [by Alex Teboul](https://www.kaggle.com/code/alexteboul/diabetes-health-indicators-dataset-notebook), with two deliberate deviations aimed at preserving information that the original pipeline discards:

1. `_BMI5` is kept at its original decimal precision instead of being rounded to the nearest integer.
2. `_AGE80` (single-year, top-coded age) is used instead of `_AGEG5YR` (5-year age bins), providing finer resolution.

## Load Data

In [2]:
zip_files = [
    "../../data/raw/2015.zip",
]

for zip_path in zip_files:
    unzip_csv(zip_path, "../../data/processed/")

Successfully extracted files: 2015.csv


In [3]:
brfss_2015_dataset = pd.read_csv("../../data/processed/2015.csv")

In [4]:
brfss_2015_dataset

,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENUM,...,_PAREC1,_PASTAE1,_LMTACT1,_LMTWRK1,_LMTSCL1,_RFSEAT2,_RFSEAT3,_FLSHOT6,_PNEUMO2,_AIDTST3
0,1.0,1.0,b'01292015',b'01',b'29',b'2015',1200.0,2.015000e+09,2.015000e+09,1.0,...,4.0,2.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,1.0
1,1.0,1.0,b'01202015',b'01',b'20',b'2015',1100.0,2.015000e+09,2.015000e+09,1.0,...,2.0,2.0,3.0,3.0,4.0,2.0,2.0,NaN,NaN,2.0
2,1.0,1.0,b'02012015',b'02',b'01',b'2015',1200.0,2.015000e+09,2.015000e+09,1.0,...,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,9.0,NaN
3,1.0,1.0,b'01142015',b'01',b'14',b'2015',1100.0,2.015000e+09,2.015000e+09,1.0,...,4.0,2.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,9.0
4,1.0,1.0,b'01142015',b'01',b'14',b'2015',1100.0,2.015000e+09,2.015000e+09,1.0,...,4.0,2.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441451,72.0,11.0,b'12162015',b'12',b'16',b'2015',1100.0,2.015005e+09,2.015005e+09,NaN,...,4.0,2.0,2.0,2.0,3.0,1.0,1.0,2.0,2.0,2.0
441452,72.0,11.0,b'12142015',b'12',b'14',b'2015',1100.0,2.015005e+09,2.015005e+09,NaN,...,2.0,2.0,3.0,3.0,4.0,1.0,1.0,NaN,NaN,1.0
441453,72.0,11.0,b'12232015',b'12',b'23',b'2015',1200.0,2.015005e+09,2.015005e+09,NaN,...,9.0,9.0,3.0,3.0,4.0,9.0,9.0,9.0,9.0,NaN
441454,72.0,11.0,b'12152015',b'12',b'15',b'2015',1100.0,2.015005e+09,2.015005e+09,NaN,...,4.0,2.0,3.0,3.0,4.0,1.0,1.0,NaN,NaN,2.0


## Feature Selection

21 features plus the target variable (`DIABETE3`) are selected based on established diabetes risk factors, following the same selection as the original methodology. `_AGEG5YR` is temporarily included alongside `_AGE80` — it is used only to filter out respondents with unreported age, then dropped (see below).

In [5]:
brfss_df_selected = brfss_2015_dataset[[
    "DIABETE3",
    "_RFHYPE5",
    "TOLDHI2",
    "_CHOLCHK",
    "_BMI5",
    "SMOKE100",
    "CVDSTRK3",
    "_MICHD",
    "_TOTINDA",
    "_FRTLT1",
    "_VEGLT1",
    "_RFDRHV5",
    "HLTHPLN1",
    "MEDCOST",
    "GENHLTH",
    "MENTHLTH",
    "PHYSHLTH",
    "DIFFWALK",
    "SEX",
    "_AGEG5YR",
    "_AGE80",
    "EDUCA",
    "INCOME2"
]]

Rows with native missing values (`NaN`) in the selected columns are removed. This accounts for the majority of the initial row reduction.

In [6]:
brfss_df_selected = brfss_df_selected.dropna()
brfss_df_selected.shape

(343606, 23)

Rows where the respondent did not report their age (`_AGEG5YR == 14`, i.e. don't know/refused/missing) are therefore removed before using `_AGE80`, ensuring only reported — not imputed — ages are retained. `_AGEG5YR` is dropped afterward, as it is no longer needed once this filtering is complete.

In [7]:
brfss_df_selected = brfss_df_selected[brfss_df_selected._AGEG5YR != 14]
brfss_df_selected = brfss_df_selected.drop(columns=["_AGEG5YR"])

`DIABETE3` is recoded into an ordinal `Diabetes_012` variable: 0 = no diabetes or diabetes only during pregnancy, 1 = prediabetes/borderline diabetes, 2 = diabetes. Rows with "don't know" (7) or "refused" (9) responses are removed.

In [8]:
brfss_df_selected["DIABETE3"] = brfss_df_selected["DIABETE3"].replace({2:0, 3:0, 1:2, 4:1})
brfss_df_selected = brfss_df_selected[brfss_df_selected.DIABETE3 != 7]
brfss_df_selected = brfss_df_selected[brfss_df_selected.DIABETE3 != 9]
brfss_df_selected.DIABETE3.unique()

array([0., 2., 1.])

Each remaining feature is individually recoded to a binary (0/1) or ordinal scale, matching the coding conventions of the original dataset. Rows containing survey sentinel codes (7 = don't know, 9 = refused, 77/99 for day-count questions) are removed via listwise deletion.

In [9]:
brfss_df_selected["_RFHYPE5"] = brfss_df_selected["_RFHYPE5"].replace({1:0, 2:1})
brfss_df_selected = brfss_df_selected[brfss_df_selected._RFHYPE5 != 9]
brfss_df_selected._RFHYPE5.unique()

array([1., 0.])

In [10]:
brfss_df_selected["TOLDHI2"] = brfss_df_selected["TOLDHI2"].replace({2:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected.TOLDHI2 != 7]
brfss_df_selected = brfss_df_selected[brfss_df_selected.TOLDHI2 != 9]
brfss_df_selected.TOLDHI2.unique()

array([1., 0.])

In [11]:
brfss_df_selected["_CHOLCHK"] = brfss_df_selected["_CHOLCHK"].replace({3:0,2:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected._CHOLCHK != 9]
brfss_df_selected._CHOLCHK.unique()

array([1., 0.])

`_BMI5` (originally BMI × 100) is divided by 100 but **not** rounded to the nearest integer, preserving decimal precision. The original methodology rounds this value, resulting in unnecessary information loss and a more coarsely discretized BMI distribution.

In [12]:
brfss_df_selected["_BMI5"] = brfss_df_selected["_BMI5"].div(100)
brfss_df_selected._BMI5.unique()

array([40.18, 25.09, 28.19, ..., 41.59, 14.44, 60.76], shape=(3527,))

In [13]:
brfss_df_selected["SMOKE100"] = brfss_df_selected["SMOKE100"].replace({2:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected.SMOKE100 != 7]
brfss_df_selected = brfss_df_selected[brfss_df_selected.SMOKE100 != 9]
brfss_df_selected.SMOKE100.unique()

array([1., 0.])

In [14]:
brfss_df_selected["CVDSTRK3"] = brfss_df_selected["CVDSTRK3"].replace({2:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected.CVDSTRK3 != 7]
brfss_df_selected = brfss_df_selected[brfss_df_selected.CVDSTRK3 != 9]
brfss_df_selected.CVDSTRK3.unique()

array([0., 1.])

In [15]:
brfss_df_selected["_MICHD"] = brfss_df_selected["_MICHD"].replace({2: 0})
brfss_df_selected._MICHD.unique()

array([0., 1.])

In [16]:
brfss_df_selected["_TOTINDA"] = brfss_df_selected["_TOTINDA"].replace({2:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected._TOTINDA != 9]
brfss_df_selected._TOTINDA.unique()

array([0., 1.])

In [17]:
brfss_df_selected["_FRTLT1"] = brfss_df_selected["_FRTLT1"].replace({2:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected._FRTLT1 != 9]
brfss_df_selected._FRTLT1.unique()

array([0., 1.])

In [18]:
brfss_df_selected["_VEGLT1"] = brfss_df_selected["_VEGLT1"].replace({2:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected._VEGLT1 != 9]
brfss_df_selected._VEGLT1.unique()

array([1., 0.])

In [19]:
brfss_df_selected["_RFDRHV5"] = brfss_df_selected["_RFDRHV5"].replace({1:0, 2:1})
brfss_df_selected = brfss_df_selected[brfss_df_selected._RFDRHV5 != 9]
brfss_df_selected._RFDRHV5.unique()

array([0., 1.])

In [20]:
brfss_df_selected["HLTHPLN1"] = brfss_df_selected["HLTHPLN1"].replace({2:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected.HLTHPLN1 != 7]
brfss_df_selected = brfss_df_selected[brfss_df_selected.HLTHPLN1 != 9]
brfss_df_selected.HLTHPLN1.unique()

array([1., 0.])

In [21]:
brfss_df_selected["MEDCOST"] = brfss_df_selected["MEDCOST"].replace({2:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected.MEDCOST != 7]
brfss_df_selected = brfss_df_selected[brfss_df_selected.MEDCOST != 9]
brfss_df_selected.MEDCOST.unique()

array([0., 1.])

In [22]:
brfss_df_selected = brfss_df_selected[brfss_df_selected.GENHLTH != 7]
brfss_df_selected = brfss_df_selected[brfss_df_selected.GENHLTH != 9]
brfss_df_selected.GENHLTH.unique()

array([5., 3., 2., 4., 1.])

In [23]:
brfss_df_selected["MENTHLTH"] = brfss_df_selected["MENTHLTH"].replace({88:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected.MENTHLTH != 77]
brfss_df_selected = brfss_df_selected[brfss_df_selected.MENTHLTH != 99]
brfss_df_selected.MENTHLTH.unique()

array([18.,  0., 30.,  3.,  5., 15., 10.,  6., 20.,  2., 25.,  1., 29.,
        4.,  7.,  8., 21., 14., 26.,  9., 16., 28., 11., 12., 24., 17.,
       13., 23., 27., 19., 22.])

In [24]:
brfss_df_selected["PHYSHLTH"] = brfss_df_selected["PHYSHLTH"].replace({88:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected.PHYSHLTH != 77]
brfss_df_selected = brfss_df_selected[brfss_df_selected.PHYSHLTH != 99]
brfss_df_selected.PHYSHLTH.unique()

array([15.,  0., 30.,  2., 14., 28.,  7., 20.,  3., 10.,  1.,  5., 17.,
        4., 19.,  6., 21., 12.,  8., 25., 27., 22., 29., 24.,  9., 16.,
       18., 23., 13., 26., 11.])

In [25]:
brfss_df_selected["DIFFWALK"] = brfss_df_selected["DIFFWALK"].replace({2:0})
brfss_df_selected = brfss_df_selected[brfss_df_selected.DIFFWALK != 7]
brfss_df_selected = brfss_df_selected[brfss_df_selected.DIFFWALK != 9]
brfss_df_selected.DIFFWALK.unique()

array([1., 0.])

In [26]:
brfss_df_selected["SEX"] = brfss_df_selected["SEX"].replace({2:0})
brfss_df_selected.SEX.unique()

array([0., 1.])

In [27]:
brfss_df_selected._AGE80.unique()

array([63., 52., 73., 70., 68., 62., 80., 58., 51., 71., 37., 47., 69.,
       54., 25., 38., 76., 75., 42., 40., 78., 55., 79., 49., 74., 77.,
       60., 59., 35., 36., 67., 64., 57., 50., 65., 61., 66., 53., 48.,
       72., 56., 22., 45., 19., 32., 43., 39., 31., 46., 20., 41., 44.,
       27., 34., 33., 30., 23., 28., 29., 21., 26., 24., 18.])

In [28]:
brfss_df_selected = brfss_df_selected[brfss_df_selected.EDUCA != 9]
brfss_df_selected.EDUCA.unique()

array([4., 6., 3., 5., 2., 1.])

In [29]:
brfss_df_selected = brfss_df_selected[brfss_df_selected.INCOME2 != 77]
brfss_df_selected = brfss_df_selected[brfss_df_selected.INCOME2 != 99]
brfss_df_selected.INCOME2.unique()

array([3., 1., 8., 6., 4., 7., 2., 5.])

The cleaned dataset retains the same row count as the original methodology (253,680 rows and 22 cols), since the deviations made here affect feature precision, not the filtering criteria that determine which rows are removed.

In [30]:
brfss_df_selected.shape

(253680, 22)

## Make feature names more readable

Raw BRFSS variable codes are renamed to descriptive names for readability.

In [31]:
brfss = brfss_df_selected.rename(columns = {
    "DIABETE3": "Diabetes_012",
    "_RFHYPE5": "HighBloodPressure",
    "TOLDHI2": "HighCholesterol",
    "_CHOLCHK": "CholCheckedRecently",
    "_BMI5": "BMI",
    "SMOKE100": "SmokedAtLeast100Cigarettes",
    "CVDSTRK3": "HadStroke",
    "_MICHD": "HeartDiseaseOrAttack",
    "_TOTINDA": "PhysicallyActive",
    "_FRTLT1": "ConsumesFruitDaily",
    "_VEGLT1": "ConsumesVeggiesDaily",
    "_RFDRHV5": "HeavyAlcoholConsumption",
    "HLTHPLN1": "HasHealthcareCoverage",
    "MEDCOST": "SkippedDoctorDueToCost",
    "GENHLTH": "GeneralHealth",
    "MENTHLTH": "MentalHealthBadDays",
    "PHYSHLTH": "PhysicalHealthBadDays",
    "DIFFWALK": "DifficultyWalking",
    "SEX": "Sex",
    "_AGE80": "Age",
    "EDUCA": "EducationLevel",
    "INCOME2": "IncomeLevel"
})

In [32]:
brfss.columns = [to_snake_case(col) for col in brfss.columns]

In [33]:
brfss

,diabetes_012,high_blood_pressure,high_cholesterol,chol_checked_recently,bmi,smoked_at_least_100_cigarettes,had_stroke,heart_disease_or_attack,physically_active,consumes_fruit_daily,...,has_healthcare_coverage,skipped_doctor_due_to_cost,general_health,mental_health_bad_days,physical_health_bad_days,difficulty_walking,sex,age,education_level,income_level
0,0.0,1.0,1.0,1.0,40.18,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,63.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.09,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,52.0,6.0,1.0
3,0.0,1.0,1.0,1.0,28.19,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,63.0,4.0,8.0
5,0.0,1.0,0.0,1.0,26.52,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,73.0,3.0,6.0
6,0.0,1.0,1.0,1.0,23.89,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,70.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441450,0.0,1.0,1.0,1.0,45.00,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,42.0,6.0,7.0
441451,2.0,1.0,1.0,1.0,18.42,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,72.0,2.0,4.0
441452,0.0,0.0,0.0,1.0,28.34,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,29.0,5.0,2.0
441454,0.0,1.0,0.0,1.0,23.15,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,52.0,5.0,1.0


In [34]:
brfss.columns

Index(['diabetes_012', 'high_blood_pressure', 'high_cholesterol',
       'chol_checked_recently', 'bmi', 'smoked_at_least_100_cigarettes',
       'had_stroke', 'heart_disease_or_attack', 'physically_active',
       'consumes_fruit_daily', 'consumes_veggies_daily',
       'heavy_alcohol_consumption', 'has_healthcare_coverage',
       'skipped_doctor_due_to_cost', 'general_health',
       'mental_health_bad_days', 'physical_health_bad_days',
       'difficulty_walking', 'sex', 'age', 'education_level', 'income_level'],
      dtype='object')

In [35]:
brfss.to_csv("../../data/processed/diabetes_012_health_indicators_BRFSS2015_clean.csv", index=False)